# Projeto Final de Recuperação — SBSE Aplicada a um Problema Concreto

**Disciplina:** Engenharia de Software Baseada em Busca na Era da IA  
**Aluno:** `<Seu Nome Completo>`  
**Problema Escolhido:** `A | B | C` (apague os que não se aplicam)  
**Data de entrega:** `<Data>`

---
> **Instruções:** Este notebook é um template. Leia os comentários `# TODO` e preencha cada seção conforme indicado.  
> Antes de entregar, execute `Kernel > Restart & Run All` e verifique que nenhuma célula retorna erro.

## 1. Identificação

Preencha os campos abaixo:

In [ ]:
# TODO: Preencha seus dados de identificação
NOME_ALUNO = "Seu Nome Completo"
PROBLEMA_ESCOLHIDO = "A"  # "A", "B" ou "C"
SEED = 42  # Mantenha esta seed em TODAS as fontes de aleatoriedade

print(f"Aluno: {NOME_ALUNO}")
print(f"Problema: {PROBLEMA_ESCOLHIDO}")
print(f"Seed global: {SEED}")

## Configuração do Ambiente (Execute primeiro — obrigatório)

In [ ]:
# @title Instalação de dependências
!pip install deap pymoo scikit-learn pandas matplotlib numpy coverage --quiet

# Forçar CPU e silenciar logs desnecessários
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import warnings
warnings.filterwarnings('ignore')

# Imports gerais
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.grid'] = True

# Fixar seeds globais
random.seed(SEED)
np.random.seed(SEED)

print("Ambiente configurado com sucesso!")

---
# ═══════════════════════════════════════════════════
# PROBLEMA A — Teste Baseado em Busca (SBST)
# Maximizando Cobertura de Ramos com Algoritmo Genético
# ═══════════════════════════════════════════════════

> **ATENÇÃO:** Execute este bloco apenas se escolheu o **Problema A**.  
> Se escolheu B ou C, role até a seção correspondente.

## 2. Problema A: Descrição e Justificativa

### 2.1. A Função-Alvo

A função `classificar_operacao` abaixo classifica operações financeiras com base em 3 parâmetros:  
- `valor` (float): valor em reais  
- `tipo` (int): 0=transferência, 1=saque, 2=investimento  
- `risco` (int): 0=baixo, 1=médio, 2=alto

A função possui **10 ramos** lógicos que precisam ser cobertos por testes. Um testador manual cobriu apenas 4.  
**Sua missão:** gerar automaticamente casos de teste que cubram o máximo de ramos possível.

### 2.2. Justificativa (TODO)

> **TODO:** Escreva aqui 2-3 parágrafos justificando por que SBSE é uma abordagem adequada  
> para este problema. Por que testar manualmente é ineficiente? Como o AG resolve isso?

In [ ]:
# @title Problema A — Função-alvo instrumentada

# Dicionário global para rastrear ramos visitados
ramos_visitados: set = set()

def classificar_operacao(valor: float, tipo: int, risco: int) -> str:
    """
    Classifica uma operação financeira.
    Instrumentada para rastrear cobertura de ramos.
    """
    global ramos_visitados

    if valor <= 0:
        ramos_visitados.add("R01_invalida")       # Ramo 1
        return "INVALIDA"
    else:
        ramos_visitados.add("R02_valor_positivo")  # Ramo 2

    if tipo == 1:  # saque
        ramos_visitados.add("R03_saque")           # Ramo 3
        if valor > 10000:
            ramos_visitados.add("R04_saque_alto")  # Ramo 4
            if risco >= 2:
                ramos_visitados.add("R05_bloqueado")   # Ramo 5
                return "BLOQUEADO"
            else:
                ramos_visitados.add("R06_monitor")     # Ramo 6
                return "APROVADO_MONITORADO"
        else:
            ramos_visitados.add("R07_saque_baixo")     # Ramo 7
            return "APROVADO"
    elif tipo == 2:  # investimento
        ramos_visitados.add("R08_investimento")        # Ramo 8
        if risco == 0 and valor >= 5000:
            ramos_visitados.add("R09_conservador")     # Ramo 9
            return "PERFIL_CONSERVADOR_OK"
        elif risco >= 1:
            ramos_visitados.add("R10_requer_analise")  # Ramo 10 — mais difícil de cobrir!
            return "REQUER_ANALISE"
        else:
            ramos_visitados.add("R11_invest_aprovado") # Ramos extras
            return "APROVADO"
    else:  # transferência
        ramos_visitados.add("R12_transferencia")       # Ramo 12
        if valor > 50000:
            ramos_visitados.add("R13_autorizacao")     # Ramo 13
            return "REQUER_AUTORIZACAO"
        ramos_visitados.add("R14_aprov_transf")        # Ramo 14
        return "APROVADO"

TOTAL_RAMOS = 10  # Os 10 ramos-alvo definidos no problema
RAMOS_ALVO = {"R01_invalida", "R02_valor_positivo", "R03_saque", "R04_saque_alto",
              "R05_bloqueado", "R06_monitor", "R07_saque_baixo", "R08_investimento",
              "R09_conservador", "R10_requer_analise"}

# Teste rápido
ramos_visitados.clear()
classificar_operacao(0, 0, 0)
classificar_operacao(500, 1, 0)
print(f"Ramos cobertos pelo teste manual inicial: {len(ramos_visitados & RAMOS_ALVO)}/{TOTAL_RAMOS}")
print(f"Ramos cobertos: {ramos_visitados & RAMOS_ALVO}")

## 3. Problema A: Formulação como Problema de SBSE

> **TODO:** Preencha a tabela abaixo com sua formulação (edite esta célula Markdown):

| Componente | Sua Definição |
| :--- | :--- |
| **Representação** | Cromossomo = ... |
| **Espaço de Busca** | valor ∈ ..., tipo ∈ ..., risco ∈ ... |
| **Função de Fitness** | ... |
| **Operadores** | Seleção: ..., Crossover: ..., Mutação: ... |
| **Critério de Parada** | ... |

## 4. Problema A: Implementação do Algoritmo Genético

In [ ]:
# @title Problema A — Setup do DEAP
from deap import base, creator, tools, algorithms

# --- Parâmetros do AG ---
# TODO: Justifique cada parâmetro em um comentário
POP_SIZE    = 30    # TODO: Por que este tamanho?
N_GEN       = 100   # TODO: Por que este número de gerações?
CX_PROB     = 0.7   # Probabilidade de crossover
MUT_PROB    = 0.2   # Probabilidade de mutação
TOURN_SIZE  = 3     # Tamanho do torneio de seleção

# Ranges dos genes [valor, tipo, risco]
GENE_MIN = [0,      0, 0]
GENE_MAX = [100001, 2, 2]

# --- Criação dos tipos DEAP ---
# Remove definições anteriores para permitir re-execução da célula
if "FitnessMax" in dir(creator): del creator.FitnessMax
if "Individual" in dir(creator): del creator.Individual

creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximizar
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

def criar_individuo():
    """Cria um caso de teste como [valor, tipo, risco]."""
    valor = random.randint(GENE_MIN[0], GENE_MAX[0])
    tipo  = random.randint(GENE_MIN[1], GENE_MAX[1])
    risco = random.randint(GENE_MIN[2], GENE_MAX[2])
    return creator.Individual([valor, tipo, risco])

toolbox.register("individual", criar_individuo)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

print("Toolbox configurado!")
print("Exemplo de indivíduo:", toolbox.individual())

In [ ]:
# @title Problema A — Função de Fitness

def avaliar_cobertura_populacao(populacao: list) -> None:
    """
    Avalia a cobertura de ramos da POPULAÇÃO INTEIRA.
    A fitness de cada indivíduo é o número de ramos cobertos por ele
    mais todos os iindivíduos anteriores (fitness acumulada).
    """
    global ramos_visitados
    ramos_visitados_antes = set(ramos_visitados)

    for ind in populacao:
        valor, tipo, risco = int(ind[0]), int(ind[1]), int(ind[2])
        # Garante que os valores estão dentro dos limites
        tipo  = max(0, min(2, tipo))
        risco = max(0, min(2, risco))
        classificar_operacao(valor, tipo, risco)

    cobertura_total = len(ramos_visitados & RAMOS_ALVO)

    for ind in populacao:
        # TODO: Implemente uma fitness individual que capture a contribuição de cada indivíduo
        # Dica: execute apenas este indivíduo em um conjunto de ramos limpo e veja quantos novos ramos ele descobre
        ind.fitness.values = (cobertura_total,)


def avaliar_individuo(individuo) -> tuple:
    """
    Função de fitness individual para uso com toolbox.
    Retorna quantos RAMOS-ALVO este indivíduo específico cobre.
    """
    global ramos_visitados
    ramos_antes = set(ramos_visitados)

    valor, tipo, risco = int(individuo[0]), int(individuo[1]), int(individuo[2])
    tipo  = max(0, min(2, tipo))
    risco = max(0, min(2, risco))
    classificar_operacao(valor, tipo, risco)

    novos_ramos = len((ramos_visitados - ramos_antes) & RAMOS_ALVO)
    return (novos_ramos,)


# Registrar operadores
toolbox.register("evaluate", avaliar_individuo)
toolbox.register("mate",     tools.cxUniform, indpb=0.5)
toolbox.register("mutate",   tools.mutUniformInt, low=GENE_MIN, up=GENE_MAX, indpb=0.4)
toolbox.register("select",   tools.selTournament, tournsize=TOURN_SIZE)

print("Operadores registrados!")

In [ ]:
# @title Problema A — Loop Evolutivo Principal

random.seed(SEED)
np.random.seed(SEED)
ramos_visitados.clear()

populacao = toolbox.population(n=POP_SIZE)

historico_cobertura_ag = []
historico_ramos_ag     = []
total_avaliacoes_ag    = 0

# Avaliação inicial
for ind in populacao:
    ind.fitness.values = toolbox.evaluate(ind)
total_avaliacoes_ag += len(populacao)

for geracao in range(N_GEN):
    # Seleção
    filhos = toolbox.select(populacao, len(populacao))
    filhos = list(map(toolbox.clone, filhos))

    # Crossover
    for filho1, filho2 in zip(filhos[::2], filhos[1::2]):
        if random.random() < CX_PROB:
            toolbox.mate(filho1, filho2)
            del filho1.fitness.values
            del filho2.fitness.values

    # Mutação
    for mutante in filhos:
        if random.random() < MUT_PROB:
            toolbox.mutate(mutante)
            del mutante.fitness.values

    # Reavaliação dos indivíduos modificados
    invalidos = [f for f in filhos if not f.fitness.valid]
    for ind in invalidos:
        ind.fitness.values = toolbox.evaluate(ind)
    total_avaliacoes_ag += len(invalidos)

    populacao[:] = filhos

    # Rastrear progresso
    cobertura_atual = len(ramos_visitados & RAMOS_ALVO)
    historico_cobertura_ag.append(cobertura_atual)
    historico_ramos_ag.append(set(ramos_visitados & RAMOS_ALVO))

    if cobertura_atual == TOTAL_RAMOS:
        print(f"Cobertura 100% atingida na geração {geracao + 1}!")
        break

cobertura_final_ag = len(ramos_visitados & RAMOS_ALVO)
print(f"\n=== Resultado do AG ===")
print(f"Cobertura final: {cobertura_final_ag}/{TOTAL_RAMOS} ({cobertura_final_ag/TOTAL_RAMOS*100:.1f}%)")
print(f"Total de avaliações: {total_avaliacoes_ag}")
print(f"Ramos cobertos: {ramos_visitados & RAMOS_ALVO}")
print(f"Ramos NÃO cobertos: {RAMOS_ALVO - ramos_visitados}")

In [ ]:
# @title Problema A — Baseline: Random Search
# IMPORTANTE: usar exatamente o mesmo orçamento de avaliações do AG

random.seed(SEED + 1)  # Seed diferente mas determinística
np.random.seed(SEED + 1)

ramos_visitados_rs = set()
historico_cobertura_rs = []
orcamento_rs = total_avaliacoes_ag  # Mesmo orçamento do AG!

for i in range(orcamento_rs):
    valor = random.randint(0, 100001)
    tipo  = random.randint(0, 2)
    risco = random.randint(0, 2)

    # Salvar estado antes
    ramos_antes = set(ramos_visitados_rs)

    # Usar versão sem ramos_visitados global
    # TODO: adapte a função para trabalhar com o conjunto ramos_visitados_rs
    # Dica: ou crie uma versão que recebe o conjunto como parâmetro
    ramos_visitados_rs.add(f"R_random_{valor}_{tipo}_{risco}"[:20])  # placeholder — substitua

    cobertura_acum = min(len(ramos_visitados_rs), TOTAL_RAMOS)
    historico_cobertura_rs.append(cobertura_acum)

cobertura_final_rs = min(len(ramos_visitados_rs), TOTAL_RAMOS)
# Saída esperada: Random Search cobriu X/10 ramos com Y avaliações
print(f"Random Search: {cobertura_final_rs}/{TOTAL_RAMOS} ramos com {orcamento_rs} avaliações")

## 6. Problema A: Resultados e Comparação

In [ ]:
# @title Problema A — Gráfico de Convergência

fig, ax = plt.subplots()

ax.plot(historico_cobertura_ag, label="Algoritmo Genético", color="royalblue", linewidth=2)
ax.axhline(y=TOTAL_RAMOS, color="green", linestyle="--", label="Cobertura máxima (10 ramos)")

# TODO: Adicione a curva do Random Search
# ax.plot(historico_cobertura_rs[:len(historico_cobertura_ag)], ...)

ax.set_xlabel("Geração")
ax.set_ylabel("Ramos cobertos")
ax.set_title("Convergência do AG — Cobertura de Ramos (Problema A)")
ax.legend()

os.makedirs("results", exist_ok=True)
plt.savefig("results/convergencia_problema_a.png", dpi=150, bbox_inches="tight")
plt.show()

# Tabela comparativa
comparativo = pd.DataFrame({
    "Método":            ["Algoritmo Genético", "Random Search"],
    "Ramos Cobertos":    [cobertura_final_ag, cobertura_final_rs],
    "Cobertura (%)": [
        f"{cobertura_final_ag/TOTAL_RAMOS*100:.1f}%",
        f"{cobertura_final_rs/TOTAL_RAMOS*100:.1f}%"
    ],
    "Avaliações":        [total_avaliacoes_ag, orcamento_rs]
})
print("\n", comparativo.to_string(index=False))
comparativo.to_csv("results/metricas_finais.csv", index=False)
print("\nResultados salvos em results/metricas_finais.csv")

## 7. Problema A: Conclusão e Análise Crítica

> **TODO:** Responda às seguintes perguntas em prosa (mínimo 3 parágrafos):
>
> 1. O AG superou o Random Search? Por que (ou por que não)?
> 2. Quais ramos foram mais difíceis de cobrir? O que isso diz sobre a estrutura da função?
> 3. Que melhorias você aplicaria se tivesse mais tempo? (ex: fitness baseada em distância de ramo, operadores adaptados)

*Escreva sua análise aqui...*

---
# ═══════════════════════════════════════════════════
# PROBLEMA B — Fairness Testing em Modelo de Crédito
# Encontrando Instâncias de Discriminação com AG
# ═══════════════════════════════════════════════════

> **ATENÇÃO:** Execute este bloco apenas se escolheu o **Problema B**.

## 2. Problema B: Descrição e Justificativa

### 2.1. O Modelo de Crédito

O modelo abaixo simula um classificador de aprovação de crédito treinado com dados históricos de um banco fictício.  
Ele recebe 5 atributos financeiros e decide se o cliente deve ter crédito aprovado ou não.  
**Suspeita:** o modelo trata de forma diferente clientes com perfis financeiros idênticos mas gênero diferente.

**Sua missão:** usar um AG para encontrar o perfil que **maximiza a diferença** de probabilidade de aprovação entre homens e mulheres.

### 2.2. Justificativa (TODO)

> **TODO:** Escreva 2-3 parágrafos sobre por que a busca exaustiva não é viável e como o AG pode sistematicamente explorar o espaço de perfis.

In [ ]:
# @title Problema B — Modelo de Crédito pré-treinado (simulado)
# Este modelo simula um classificador treinado com viés de gênero

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np

# Gera dados sintéticos com viés embutido
np.random.seed(SEED)
n_amostras = 2000

# Gênero: 0=F, 1=M (coeficiente positivo → homens com vantagem)
genero    = np.random.randint(0, 2, n_amostras)
idade     = np.random.randint(18, 70, n_amostras).astype(float)
renda     = np.random.exponential(scale=5000, size=n_amostras) + 1000
divida    = np.random.exponential(scale=3000, size=n_amostras)
score     = np.random.randint(300, 900, n_amostras).astype(float)
tempo_emp = np.random.randint(0, 30, n_amostras).astype(float)

# Aprovação: score + renda dominam, mas gênero tem peso oculto (viés)
logit    = (-4
            + 0.006 * score
            + 0.0001 * renda
            - 0.0002 * divida
            + 0.02   * tempo_emp
            + 0.8    * genero    # <- VIÉS: homens têm vantagem artificial
            + np.random.normal(0, 0.5, n_amostras))
aprovado = (logit > 0).astype(int)

# Features: age, renda, divida, score, tempo_emprego, genero
X = np.column_stack([idade, renda, divida, score, tempo_emp, genero])
y = aprovado

# Treinar modelo (sem remover a coluna genero — isso simula negligência real)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED)

modelo_credito = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(random_state=SEED, max_iter=500))
])
modelo_credito.fit(X_train, y_train)

acc = modelo_credito.score(X_test, y_test)
print(f"Modelo treinado. Acurácia no teste: {acc:.2%}")
print("Formato de entrada: [idade, renda_mensal, divida_total, score_credito, tempo_emprego, genero]")
print("Genero: 0=Feminino, 1=Masculino")

## 3. Problema B: Formulação como Problema de SBSE

> **TODO:** Preencha abaixo com sua formulação:

| Componente | Sua Definição |
| :--- | :--- |
| **Representação** | Cromossomo = `[idade, renda, divida, score, tempo_emprego]` (5 genes reais — genero é fixo na avaliação) |
| **Fitness** | $f = \|P(\text{aprovado}\|\text{perfil, genero=M}) - P(\text{aprovado}\|\text{perfil, genero=F})\|$ |
| **Operadores** | Seleção: ..., Crossover: ..., Mutação: ... |
| **Critério de Parada** | ... |

In [ ]:
# @title Problema B — Setup do AG (DEAP)
from deap import base, creator, tools

# Ranges de cada gene [idade, renda, divida, score, tempo_emprego]
B_GENE_MIN  = [18,    500,   0,    300, 0]
B_GENE_MAX  = [70,  30000, 50000, 850, 40]

# Parâmetros do AG
B_POP_SIZE  = 50
B_N_GEN     = 200
B_CX_PROB   = 0.7
B_MUT_PROB  = 0.2
B_SIGMA     = [2, 500, 500, 25, 1]  # desvio padrão por gene para mutGaussian

if "FitnessMax" in dir(creator): del creator.FitnessMax
if "Individual" in dir(creator): del creator.Individual

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

b_toolbox = base.Toolbox()

def criar_perfil():
    """Cria um perfil financeiro aleatório (sem o atributo gênero)."""
    return creator.Individual([
        float(random.randint(int(B_GENE_MIN[0]), int(B_GENE_MAX[0]))),   # idade
        random.uniform(B_GENE_MIN[1], B_GENE_MAX[1]),   # renda
        random.uniform(B_GENE_MIN[2], B_GENE_MAX[2]),   # divida
        float(random.randint(int(B_GENE_MIN[3]), int(B_GENE_MAX[3]))),   # score
        float(random.randint(int(B_GENE_MIN[4]), int(B_GENE_MAX[4]))),   # tempo_emprego
    ])

b_toolbox.register("individual", criar_perfil)
b_toolbox.register("population", tools.initRepeat, list, b_toolbox.individual)

print("Toolbox B configurado!")
print("Exemplo de perfil:", b_toolbox.individual())

In [ ]:
# @title Problema B — Função de Fitness e Operadores

def avaliar_discriminacao(perfil) -> tuple:
    """
    Calcula a disparidade de predição entre gênero M e F para um dado perfil.
    Retorna a diferença absoluta de probabilidade de aprovação.
    """
    # TODO: Implemente usando modelo_credito.predict_proba()
    # Dica: crie dois arrays — perfil_m (genero=1) e perfil_f (genero=0)
    # e use predict_proba para obter P(aprovado)
    perfil_m = np.array(list(perfil) + [1]).reshape(1, -1)  # genero=Masculino
    perfil_f = np.array(list(perfil) + [0]).reshape(1, -1)  # genero=Feminino

    prob_m = modelo_credito.predict_proba(perfil_m)[0][1]  # P(aprovado | M)
    prob_f = modelo_credito.predict_proba(perfil_f)[0][1]  # P(aprovado | F)

    discriminacao = abs(prob_m - prob_f)
    return (discriminacao,)


# Mutação Gaussiana com clipping para manter valores nos limites
def mutar_perfil(individuo, sigma, indpb):
    """Mutação Gaussiana com clipping para manter genes nos limites definidos."""
    tools.mutGaussian(individuo, mu=0, sigma=sigma, indpb=indpb)
    for i in range(len(individuo)):
        individuo[i] = max(B_GENE_MIN[i], min(B_GENE_MAX[i], individuo[i]))
    return (individuo,)


b_toolbox.register("evaluate", avaliar_discriminacao)
b_toolbox.register("mate",     tools.cxBlend, alpha=0.5)
b_toolbox.register("mutate",   mutar_perfil, sigma=B_SIGMA, indpb=0.3)
b_toolbox.register("select",   tools.selTournament, tournsize=3)

# Teste da função de fitness
perfil_teste = b_toolbox.individual()
fit_teste = avaliar_discriminacao(perfil_teste)
print(f"Fitness de perfil aleatório: {fit_teste[0]:.4f} (diferença de probabilidade)")

In [ ]:
# @title Problema B — Loop Evolutivo e Baseline
from deap import algorithms

random.seed(SEED)
np.random.seed(SEED)

b_pop = b_toolbox.population(n=B_POP_SIZE)
for ind in b_pop:
    ind.fitness.values = b_toolbox.evaluate(ind)

b_historico = []
b_total_aval = B_POP_SIZE

for gen in range(B_N_GEN):
    # TODO: Implemente o loop evolutivo completo
    # (seleção, crossover, mutação, reavaliação)
    # Pode usar algorithms.eaSimple como alternativa
    filhos = b_toolbox.select(b_pop, len(b_pop))
    filhos = list(map(b_toolbox.clone, filhos))

    for f1, f2 in zip(filhos[::2], filhos[1::2]):
        if random.random() < B_CX_PROB:
            b_toolbox.mate(f1, f2)
            del f1.fitness.values; del f2.fitness.values

    for mutante in filhos:
        if random.random() < B_MUT_PROB:
            b_toolbox.mutate(mutante)
            del mutante.fitness.values

    invalidos = [f for f in filhos if not f.fitness.valid]
    for ind in invalidos:
        ind.fitness.values = b_toolbox.evaluate(ind)
    b_total_aval += len(invalidos)

    b_pop[:] = filhos
    melhor = max(b_pop, key=lambda x: x.fitness.values[0])
    b_historico.append(melhor.fitness.values[0])

melhor_ag = max(b_pop, key=lambda x: x.fitness.values[0])
print(f"\n=== Melhor perfil discriminatório encontrado ===")
print(f"Disparidade: {melhor_ag.fitness.values[0]:.4f} ({melhor_ag.fitness.values[0]*100:.1f} p.p.)")
print(f"Perfil: Idade={melhor_ag[0]:.0f}, Renda=R${melhor_ag[1]:.0f}, Dívida=R${melhor_ag[2]:.0f}, Score={melhor_ag[3]:.0f}, Tempo={melhor_ag[4]:.0f} anos")

# Baseline Random Search
random.seed(SEED + 1)
np.random.seed(SEED + 1)
b_rs_historico = []
for _ in range(b_total_aval):
    ind_rs = b_toolbox.individual()
    fit_rs = avaliar_discriminacao(ind_rs)
    b_rs_historico.append(fit_rs[0])

melhor_rs_fitness = max(b_rs_historico)
print(f"\nRandom Search — melhor disparidade: {melhor_rs_fitness:.4f} ({b_total_aval} avaliações)")

In [ ]:
# @title Problema B — Gráfico de Convergência e Top-5 perfis

# Gráfico de convergência
fig, ax = plt.subplots()
ax.plot(b_historico, label="AG (melhor por geração)", color="royalblue", linewidth=2)
# TODO: adicione a curva do Random Search (máximo acumulado)
ax.set_xlabel("Geração / Avaliação")
ax.set_ylabel("Disparidade de Probabilidade")
ax.set_title("Convergência — Fairness Testing (Problema B)")
ax.legend()
os.makedirs("results", exist_ok=True)
plt.savefig("results/convergencia_problema_b.png", dpi=150, bbox_inches="tight")
plt.show()

# Top-5 perfis mais discriminatórios
b_pop_ordenada = sorted(b_pop, key=lambda x: x.fitness.values[0], reverse=True)
registros = []
for i, ind in enumerate(b_pop_ordenada[:5]):
    pf_m = np.array(list(ind) + [1]).reshape(1, -1)
    pf_f = np.array(list(ind) + [0]).reshape(1, -1)
    prob_m = modelo_credito.predict_proba(pf_m)[0][1]
    prob_f = modelo_credito.predict_proba(pf_f)[0][1]
    registros.append({
        "Rank": i+1,
        "Idade": f"{ind[0]:.0f}",
        "Renda (R$)": f"{ind[1]:.0f}",
        "Dívida (R$)": f"{ind[2]:.0f}",
        "Score": f"{ind[3]:.0f}",
        "Tempo (anos)": f"{ind[4]:.0f}",
        "P(aprov|M)": f"{prob_m:.3f}",
        "P(aprov|F)": f"{prob_f:.3f}",
        "Disparidade": f"{abs(prob_m - prob_f):.3f}"
    })

df_top5 = pd.DataFrame(registros)
print("\nTop-5 Perfis Mais Discriminatórios:")
print(df_top5.to_string(index=False))
df_top5.to_csv("results/metricas_finais.csv", index=False)

## 7. Problema B: Conclusão, Análise Crítica e Questão Ética

> **TODO (obrigatório):** Responda às seguintes perguntas:
>
> 1. O AG encontrou perfis com disparidade relevante? Em que faixa de renda o modelo é mais discriminatório?
> 2. O AG superou o Random Search? Em que proporção?
> 3. **Questão Ética:** Você encontrou viés de verdade ou o modelo aprendeu uma correlação estatística legítima? Justifique em pelo menos 3 parágrafos.

*Escreva sua análise aqui...*

---
# ═══════════════════════════════════════════════════
# PROBLEMA C — Otimização de Hiperparâmetros
# Random Forest com Evolução Diferencial (DEAP)
# ═══════════════════════════════════════════════════

> **ATENÇÃO:** Execute este bloco apenas se escolheu o **Problema C**.

## 2. Problema C: Descrição e Justificativa

### 2.1. O Dataset e o Problema

Usaremos o **Wisconsin Breast Cancer Dataset** — tarefa de classificação binária (maligno vs. benigno).  
O Grid Search padrão (fornecido no template) atingiu **94,7% de acurácia**.  
**Sua missão:** superar esse resultado usando Evolução Diferencial para otimizar os hiperparâmetros do `RandomForestClassifier`.

### 2.2. Justificativa (TODO)

> **TODO:** Por que Grid Search é ineficiente para espaços de busca contínuos e de alta dimensão? Como a Evolução Diferencial endereça esse problema?

In [ ]:
# @title Problema C — Dataset e Baseline (Grid Search)
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import time

# Carregar e separar dados (NUNCA use X_test durante a otimização)
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.25, random_state=SEED, stratify=y_bc)

# Normalização (fit APENAS no treino)
scaler_bc = StandardScaler()
X_tr_s = scaler_bc.fit_transform(X_tr)
X_te_s = scaler_bc.transform(X_te)

print(f"Treino: {X_tr_s.shape} | Teste: {X_te_s.shape}")

# --- Baseline: Grid Search ---
grid_params = {
    "n_estimators": [50, 100, 200],
    "max_depth":    [5, 10, None],
    "max_features": ["sqrt", "log2"]
}
t0 = time.time()
gs = GridSearchCV(RandomForestClassifier(random_state=SEED), grid_params, cv=5, n_jobs=-1)
gs.fit(X_tr_s, y_tr)
tempo_grid = time.time() - t0

y_pred_gs = gs.predict(X_te_s)
acc_gs = accuracy_score(y_te, y_pred_gs)
f1_gs  = f1_score(y_te, y_pred_gs)

print(f"\n=== Grid Search ===")
print(f"Melhores hiperparâmetros: {gs.best_params_}")
print(f"Acurácia (teste): {acc_gs:.4f} ({acc_gs*100:.2f}%)")
print(f"F1-Score  (teste): {f1_gs:.4f}")
print(f"Tempo total: {tempo_grid:.1f}s | Avaliações: {len(gs.cv_results_['params'])*5}")

## 3. Problema C: Formulação como Problema de SBSE

> **TODO:** Preencha a tabela com sua formulação:

| Componente | Sua Definição |
| :--- | :--- |
| **Cromossomo** | `[n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features_pct]` |
| **Espaço de Busca** | `n_estimators ∈ [10, 500]`, `max_depth ∈ [2, 30]`, ... |
| **Fitness** | Negativo da acurácia média em 5-fold CV no conjunto de **treino** |
| **Algoritmo** | Evolução Diferencial (DEAP) |
| **Critério de Parada** | ... |

In [ ]:
# @title Problema C — Setup DEAP: Evolução Diferencial
from deap import base, creator, tools

# Ranges: [n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features_pct]
C_GENE_MIN = [10,   2,  2, 1, 0.1]
C_GENE_MAX = [500, 30, 20, 10, 1.0]

# Parâmetros da ED
C_POP_SIZE  = 20   # Populações menores funcionam bem em ED
C_N_GEN     = 50
C_CR        = 0.7  # Taxa de crossover binomial
C_F         = 0.8  # Fator de diferenciação

if "FitnessMin" in dir(creator): del creator.FitnessMin
if "Individual" in dir(creator): del creator.Individual

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # Minimizar (acurácia negada)
creator.create("Individual", list, fitness=creator.FitnessMin)

c_toolbox = base.Toolbox()

def criar_hparams():
    """Cria um vetor de hiperparâmetros dentro dos limites definidos."""
    return creator.Individual(
        [random.uniform(C_GENE_MIN[i], C_GENE_MAX[i]) for i in range(len(C_GENE_MIN))]
    )

c_toolbox.register("individual", criar_hparams)
c_toolbox.register("population", tools.initRepeat, list, c_toolbox.individual)

print("Toolbox C configurado!")
print("Exemplo de cromossomo:", [f"{x:.2f}" for x in c_toolbox.individual()])

In [ ]:
# @title Problema C — Função de Fitness (5-fold CV)

def decodificar_hparams(cromossomo: list) -> dict:
    """Converte o vetor real contínuo para hiperparâmetros do RandomForest."""
    n_est     = int(round(cromossomo[0]))
    max_d     = int(round(cromossomo[1]))
    min_split = int(round(cromossomo[2]))
    min_leaf  = int(round(cromossomo[3]))
    max_feat  = float(cromossomo[4])  # fração de features
    return {
        "n_estimators":      max(10, n_est),
        "max_depth":         max(2, max_d),
        "min_samples_split": max(2, min_split),
        "min_samples_leaf":  max(1, min_leaf),
        "max_features":      max(0.1, min(1.0, max_feat)),
        "random_state":      SEED
    }


def avaliar_hparams(cromossomo) -> tuple:
    """Avalia hiperparâmetros via 5-fold CV no conjunto de treino. Retorna acurácia negada (DEAP minimiza)."""
    hparams = decodificar_hparams(cromossomo)
    rf = RandomForestClassifier(**hparams)
    scores = cross_val_score(rf, X_tr_s, y_tr, cv=5, scoring="accuracy", n_jobs=-1)
    return (-scores.mean(),)  # Negado pois DEAP minimiza


c_toolbox.register("evaluate", avaliar_hparams)

# Teste rápido da fitness
ind_teste = c_toolbox.individual()
fit_teste  = avaliar_hparams(ind_teste)
print(f"Fitness de cromossomo aleatório: {-fit_teste[0]:.4f} (acurácia CV)")
print(f"Hiperparâmetros: {decodificar_hparams(ind_teste)}")

In [ ]:
# @title Problema C — Evolução Diferencial (Loop Principal)
# Implementação manual do operador DE/rand/1/bin

random.seed(SEED)
np.random.seed(SEED)

c_pop = c_toolbox.population(n=C_POP_SIZE)

# Avaliar população inicial
for ind in c_pop:
    ind.fitness.values = c_toolbox.evaluate(ind)

c_historico_melhor = []
c_total_aval = C_POP_SIZE

for gen in range(C_N_GEN):
    for i, alvo in enumerate(c_pop):
        # DE/rand/1: seleciona 3 indivíduos aleatórios distintos
        candidatos = [j for j in range(C_POP_SIZE) if j != i]
        r1, r2, r3 = random.sample(candidatos, 3)
        a, b_ind, c_ind = c_pop[r1], c_pop[r2], c_pop[r3]

        # Mutação diferencial: vetor doador
        doador = [a[k] + C_F * (b_ind[k] - c_ind[k]) for k in range(len(alvo))]

        # Clipping para manter nos limites
        doador = [max(C_GENE_MIN[k], min(C_GENE_MAX[k], doador[k])) for k in range(len(doador))]

        # Crossover binomial
        j_rand = random.randint(0, len(alvo) - 1)
        trial  = [doador[k] if random.random() < C_CR or k == j_rand else alvo[k]
                  for k in range(len(alvo))]
        trial_ind = creator.Individual(trial)
        trial_ind.fitness.values = c_toolbox.evaluate(trial_ind)
        c_total_aval += 1

        # Seleção greedy: aceita trial se melhor (menor = melhor, pois minimizamos)
        if trial_ind.fitness.values[0] < alvo.fitness.values[0]:
            c_pop[i] = trial_ind

    melhor_gen = min(c_pop, key=lambda x: x.fitness.values[0])
    c_historico_melhor.append(-melhor_gen.fitness.values[0])  # Acurácia positiva

melhor_ed = min(c_pop, key=lambda x: x.fitness.values[0])
print(f"\n=== Resultado da Evolução Diferencial ===")
print(f"Acurácia CV (treino): {-melhor_ed.fitness.values[0]:.4f}")
print(f"Hiperparâmetros: {decodificar_hparams(melhor_ed)}")
print(f"Total de avaliações: {c_total_aval}")

In [ ]:
# @title Problema C — Avaliação Final no Conjunto de Teste + Comparação

# Treinar modelo final com melhores hiperparâmetros e avaliar no TESTE
hparams_finais = decodificar_hparams(melhor_ed)
rf_final = RandomForestClassifier(**hparams_finais)
rf_final.fit(X_tr_s, y_tr)
y_pred_ed = rf_final.predict(X_te_s)
acc_ed = accuracy_score(y_te, y_pred_ed)
f1_ed  = f1_score(y_te, y_pred_ed)

# Gráfico de convergência
fig, ax = plt.subplots()
ax.plot(c_historico_melhor, label="Evolução Diferencial", color="royalblue", linewidth=2)
ax.axhline(y=acc_gs, color="orange", linestyle="--", linewidth=2, label=f"Grid Search ({acc_gs:.4f})")
ax.set_xlabel("Geração")
ax.set_ylabel("Acurácia CV (treino)")
ax.set_title("Convergência — Otimização de Hiperparâmetros (Problema C)")
ax.legend()
os.makedirs("results", exist_ok=True)
plt.savefig("results/convergencia_problema_c.png", dpi=150, bbox_inches="tight")
plt.show()

# Tabela comparativa
comparativo_c = pd.DataFrame({
    "Método":             ["Grid Search", "Evolução Diferencial"],
    "Acurácia (teste)":  [f"{acc_gs:.4f}", f"{acc_ed:.4f}"],
    "F1-Score (teste)":  [f"{f1_gs:.4f}",  f"{f1_ed:.4f}"],
    "Avaliações":         [len(gs.cv_results_['params'])*5, c_total_aval]
})
print("\n", comparativo_c.to_string(index=False))
comparativo_c.to_csv("results/metricas_finais.csv", index=False)
print("\nResultados salvos em results/metricas_finais.csv")

## 7. Problema C: Conclusão e Análise Crítica

> **TODO:** Responda às seguintes perguntas:
>
> 1. A ED superou o Grid Search em acurácia? Com quantas avaliações a menos (ou a mais)?
> 2. Qual hiperparâmetro parece ter maior impacto na acurácia? Justifique com base nos resultados.
> 3. O que poderia ser melhorado nesta abordagem? (ex: adaptive CR/F, restart automático, multi-objetivo)

*Escreva sua análise aqui...*

---
## Bônus: Integração com LLM (+0,5 pontos — máx. nota 7,0)

Escolha **uma** das seguintes integrações com LLM e implemente abaixo:

**Opção 1 (Problema A):** Use um LLM para, dado um ramo não coberto, sugerir valores de entrada que o cubram. Compare se as sugestões do LLM são melhores que indivíduos aleatórios como ponto de partida.

**Opção 2 (Problema B):** Use um LLM para analisar o top-5 de perfis discriminatórios e produzir uma hipótese sobre qual subgrupo populacional é mais afetado.

**Opção 3 (Problema C):** Use um LLM para, dado o histórico de avaliações, sugerir novos hiperparâmetros a tentar na próxima geração (LLM como operador de crossover verbal).

> **TODO:** Implemente a integração aqui. Documente por que a integração é relevante (não cosmética).

In [ ]:
# @title Bônus — Integração com LLM (opcional)
# TODO: Implemente aqui sua integração com LLM
# Exemplo de estrutura usando OpenAI:

# from openai import OpenAI
# client = OpenAI(api_key="SUA_API_KEY")
# 
# def consultar_llm(contexto: str, pergunta: str) -> str:
#     resposta = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": "Você é um especialista em SBSE e fairness em IA."},
#             {"role": "user",   "content": f"{contexto}\n\n{pergunta}"}
#         ]
#     )
#     return resposta.choices[0].message.content

print("Bônus não implementado — seção disponível para integração com LLM.")